# Notebook 09 — Final Pipeline Export
## Section 1: Save Final Model on Disk

Load `final_model_selection.json` from Notebook 08, retrain the **selected** model
(Random Forest / GradientBoosting / XGBoost / LightGBM / linear), evaluate once on
September test, and save as `final_model.joblib`.

**Metric:** MAE (minutes)

In [1]:
# =============================================================================
# Notebook 09 | Section 1: Save Final Model on Disk
# =============================================================================
# Load final_model_selection.json → retrain selected model → save joblib.
# =============================================================================

from __future__ import annotations

import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from xgboost import XGBRegressor
except ImportError as e:
    raise ImportError("Install xgboost: pip install xgboost") from e

try:
    from lightgbm import LGBMRegressor
except ImportError as e:
    raise ImportError("Install lightgbm: pip install lightgbm") from e

USE_TRAIN_SAMPLE = True
TRAIN_SAMPLE_SIZE = 120_000
RANDOM_STATE = 42

PRIMARY_TARGET = "delay_in_min"
CATEGORICAL_COLS = ["eva", "train_number", "final_destination_station"]


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROCESSED_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SEL_PATH = REFERENCE_DIR / "final_model_selection.json"
TRAIN_PATH = PROCESSED_DIR / "ice_train.parquet"
TEST_PATH = PROCESSED_DIR / "ice_test.parquet"

MODEL_PATH = MODELS_DIR / "final_model.joblib"
LEGACY_RF_PATH = MODELS_DIR / "final_rf_model.joblib"
EXPORT_REPORT_PATH = REFERENCE_DIR / "final_model_export_report.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def build_preprocessor(feature_cols: list[str], scale_numeric: bool) -> ColumnTransformer:
    cat_cols = [c for c in CATEGORICAL_COLS if c in feature_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    if scale_numeric:
        num_pipe = Pipeline(
            [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
        )
    else:
        num_pipe = SimpleImputer(strategy="median")

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                cat_cols,
            ),
        ]
    )


def make_estimator(algorithm: str, params: dict):
    p = dict(params or {})
    if algorithm == "RandomForestRegressor":
        p.setdefault("random_state", RANDOM_STATE)
        p.setdefault("n_jobs", -1)
        return RandomForestRegressor(**p), False
    if algorithm == "GradientBoosting":
        p.setdefault("random_state", RANDOM_STATE)
        return GradientBoostingRegressor(**p), False
    if algorithm == "XGBoost":
        p.setdefault("random_state", RANDOM_STATE)
        p.setdefault("n_jobs", -1)
        return XGBRegressor(**p), False
    if algorithm == "LightGBM":
        p.setdefault("random_state", RANDOM_STATE)
        p.setdefault("n_jobs", -1)
        p.setdefault("verbosity", -1)
        return LGBMRegressor(**p), False
    if algorithm == "LinearRegression":
        return LinearRegression(), True
    if algorithm == "Ridge":
        return Ridge(alpha=1.0, random_state=RANDOM_STATE), True
    if algorithm == "Lasso":
        return Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE), True
    if algorithm == "ElasticNet":
        return ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE), True
    raise ValueError(f"Unsupported algorithm for export: {algorithm}")


final_sel = load_json(FINAL_SEL_PATH)
fm = final_sel["final_model"]

feature_cols = fm["feature_columns"]
best_params = fm.get("best_params") or {}
algorithm = fm["algorithm"]

print("Notebook 09 | Section 1 — Save final model")
print(f"Algorithm   : {algorithm}")
print(f"Feature set : {fm['feature_set']}")
print(f"Features    : {len(feature_cols)}")
print(f"Best params : {best_params}")
print(f"Expected MAE: {fm['test_mae']:.4f} min (from NB08 Sep test)")
print()

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

sample_algos = {"GradientBoosting", "XGBoost", "LightGBM", "RandomForestRegressor"}
use_sample = USE_TRAIN_SAMPLE and algorithm in sample_algos and len(train_df) > TRAIN_SAMPLE_SIZE

if use_sample:
    n = int(fm.get("train_rows_used") or TRAIN_SAMPLE_SIZE)
    n = min(n, len(train_df))
    fit_df = train_df.sample(n=n, random_state=RANDOM_STATE)
    print(f"Training sample: {len(fit_df):,} / {len(train_df):,} train rows")
else:
    fit_df = train_df
    print(f"Training on full train: {len(fit_df):,} rows")

print(f"Test rows (verify): {len(test_df):,}")
print()

estimator, scale_numeric = make_estimator(algorithm, best_params)
final_pipeline = Pipeline(
    [
        ("prep", build_preprocessor(feature_cols, scale_numeric=scale_numeric)),
        ("model", estimator),
    ]
)

X_train = fit_df[feature_cols]
y_train = fit_df[PRIMARY_TARGET].values
X_test = test_df[feature_cols]
y_test = test_df[PRIMARY_TARGET].values

print("Training final model...")
final_pipeline.fit(X_train, y_train)

train_mae = round(float(mean_absolute_error(y_train, final_pipeline.predict(X_train))), 4)
test_mae = round(float(mean_absolute_error(y_test, final_pipeline.predict(X_test))), 4)

print(f"  Train MAE: {train_mae:.4f} min")
print(f"  Test MAE : {test_mae:.4f} min  (reference NB08: {fm['test_mae']:.4f})")
print()

joblib.dump(final_pipeline, MODEL_PATH)
joblib.dump(final_pipeline, LEGACY_RF_PATH)

export_report = {
    "metadata": {
        "notebook": "09_Final_Pipeline_Export",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
    },
    "final_model": {
        "algorithm": algorithm,
        "export_name": fm.get("export_name", algorithm),
        "feature_set": fm["feature_set"],
        "feature_columns": feature_cols,
        "best_params": best_params,
        "source_selection_file": str(FINAL_SEL_PATH),
    },
    "training": {
        "train_parquet": str(TRAIN_PATH),
        "train_months": ["2024-07", "2024-08"],
        "rows_used_for_fit": int(len(fit_df)),
        "use_train_sample": use_sample,
    },
    "evaluation_on_reload": {
        "test_parquet": str(TEST_PATH),
        "test_month": "2024-09",
        "train_mae": train_mae,
        "test_mae": test_mae,
        "nb08_reference_test_mae": fm["test_mae"],
    },
    "saved_files": {
        "model_joblib": str(MODEL_PATH),
        "legacy_model_joblib": str(LEGACY_RF_PATH),
        "export_report": str(EXPORT_REPORT_PATH),
    },
}

with EXPORT_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(export_report), f, indent=2, ensure_ascii=False)

demo = test_df.head(3)[feature_cols + [PRIMARY_TARGET]].copy()
demo["predicted_delay_min"] = final_pipeline.predict(demo[feature_cols]).round(2)

print("=" * 72)
print("Section 1 COMPLETE")
print("=" * 72)
print(f"Model saved : {MODEL_PATH}")
print(f"Also saved  : {LEGACY_RF_PATH}")
print(f"Report saved: {EXPORT_REPORT_PATH}")
print()
print("Demo predictions (first 3 test rows):")
print(demo[[PRIMARY_TARGET, "predicted_delay_min"]].to_string(index=False))
print()
print("Next: Section 2 — full pipeline summary + reproduce guide")
print("=" * 72)

Notebook 09 | Section 1 — Save final model
Algorithm   : XGBoost
Feature set : full_with_weather
Features    : 15
Best params : {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1, 'tree_method': 'hist'}
Expected MAE: 10.3111 min (from NB08 Sep test)

Training sample: 80,000 / 255,062 train rows
Test rows (verify): 121,964

Training final model...
  Train MAE: 9.8496 min
  Test MAE : 10.3111 min  (reference NB08: 10.3111)

Section 1 COMPLETE
Model saved : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\models\final_model.joblib
Also saved  : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\models\final_rf_model.joblib
Report saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\final_model_export_report.json

Demo predictions (first 3 test rows):
 delay_in_min  predicted_delay_min
            0            11.820000
           

# Notebook 09 — Final Pipeline Export
## Section 2: Full Pipeline Summary & Reproduce Guide

One master document: all notebooks, all saved Parquet files, final results, and how to reproduce.

**Output:** `full_project_pipeline_summary.json`

In [2]:
# =============================================================================
# Notebook 09 | Section 2: Full Pipeline Summary & Reproduce Guide
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROCESSED_DIR / "models"
SUMMARY_PATH = REFERENCE_DIR / "full_project_pipeline_summary.json"


def load_json(path: Path):
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def parquet_info(path: Path) -> dict:
    if not path.exists():
        return {"exists": False, "path": str(path)}
    try:
        rows = int(len(pd.read_parquet(path, columns=["delay_in_min"])))
    except Exception:
        rows = int(len(pd.read_parquet(path)))
    return {
        "exists": True,
        "path": str(path),
        "size_mb": round(path.stat().st_size / 1e6, 2),
        "rows": rows,
    }


config = load_json(REFERENCE_DIR / "project_config.json")
final_sel = load_json(REFERENCE_DIR / "final_model_selection.json")
ablation = load_json(REFERENCE_DIR / "weather_ablation_report.json")
TARGET_MONTHS = config["scope"]["target_months"]

print("Notebook 09 | Section 2 — Full pipeline summary")
print()

inventory = {
    "merged": {f"ice_weather_merged_{m}.parquet": parquet_info(PROCESSED_DIR / f"ice_weather_merged_{m}.parquet") for m in TARGET_MONTHS},
    "train_test": {
        "ice_train.parquet": parquet_info(PROCESSED_DIR / "ice_train.parquet"),
        "ice_test.parquet": parquet_info(PROCESSED_DIR / "ice_test.parquet"),
    },
}

pipeline_steps = [
    {"notebook": "01-06", "title": "Data → Features", "outputs": ["ice_train.parquet", "ice_test.parquet"]},
    {"notebook": "07", "title": "Models", "outputs": ["baseline_results.json", "random_forest_results.json", "boosting_results.json"]},
    {"notebook": "08", "title": "Tuning & Selection", "outputs": ["rf_tuning_results.json", "final_model_selection.json"]},
    {"notebook": "09", "title": "Export", "outputs": ["final_model.joblib", "full_project_pipeline_summary.json"]},
]

merged_total = sum(
    inventory["merged"][f"ice_weather_merged_{m}.parquet"]["rows"]
    for m in TARGET_MONTHS
    if inventory["merged"][f"ice_weather_merged_{m}.parquet"]["exists"]
)

fm = final_sel["final_model"]
summary = {
    "metadata": {
        "project_title": config["project"]["title"],
        "authors": config["project"]["authors"],
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "notebook": "09_Final_Pipeline_Export",
        "section": "Section 2",
        "status": "PROJECT COMPLETE",
    },
    "professor_aligned": {
        "regression_only": True,
        "target": "delay_in_min",
        "primary_metric": "mae",
        "models_compared": [
            "Naive", "Linear/Ridge/Lasso/ElasticNet",
            "RandomForest", "GradientBoosting", "XGBoost", "LightGBM",
        ],
    },
    "key_results": {
        "final_model": fm,
        "naive_median_test_mae": final_sel["benchmarks"]["naive_median_test_mae"],
        "weather_ablation_gain_min": ablation["ablation"]["weather_gain_min"] if ablation else None,
        "train_rows": inventory["train_test"]["ice_train.parquet"].get("rows"),
        "test_rows": inventory["train_test"]["ice_test.parquet"].get("rows"),
        "merged_rows_total": merged_total,
    },
    "pipeline_steps": pipeline_steps,
    "research_questions": final_sel["rq_answers"],
    "honest_conclusion": final_sel.get("honest_conclusion"),
    "reproduce_guide": {
        "step_1": "Run Notebooks 01-09 in order",
        "step_2": "pip install xgboost lightgbm",
        "step_3": "Load model: joblib.load('data/processed/models/final_model.joblib')",
    },
    "important_files_for_viva": {
        "final_model": "data/processed/models/final_model.joblib",
        "final_results": "data/reference/final_model_selection.json",
        "boosting_results": "data/reference/boosting_results.json",
    },
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

print("=" * 72)
print("KEY RESULTS")
print("=" * 72)
print(f"  Final model : {fm.get('export_name', fm['algorithm'])} ({fm['feature_set']})")
print(f"  Final MAE   : {fm['test_mae']:.4f} min")
print(f"  Naive median: {final_sel['benchmarks']['naive_median_test_mae']:.4f} min")
print()
print(f"Saved: {SUMMARY_PATH}")
print("Next: Section 3 — final close-out")
print("=" * 72)

Notebook 09 | Section 2 — Full pipeline summary

KEY RESULTS
  Final model : XGBoost (full_with_weather)
  Final MAE   : 10.3111 min
  Naive median: 9.4813 min

Saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\full_project_pipeline_summary.json
Next: Section 3 — final close-out


# Notebook 09 — Final Pipeline Export
## Section 3: Project Close-Out

Final checklist: model on disk, pipeline summary saved, project complete.

**Status:** ICE Delay Prediction capstone — DONE

In [3]:
# =============================================================================
# Notebook 09 | Section 3: Final Project Close-Out
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROCESSED_DIR / "models"
FIGURES_DIR = PROCESSED_DIR / "figures"

SUMMARY_PATH = REFERENCE_DIR / "notebook_09_summary.json"
PIPELINE_SUMMARY = REFERENCE_DIR / "full_project_pipeline_summary.json"
PROJECT_COMPLETE = REFERENCE_DIR / "project_complete.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


config = load_json(REFERENCE_DIR / "project_config.json")
pipeline = load_json(PIPELINE_SUMMARY)
final_sel = load_json(REFERENCE_DIR / "final_model_selection.json")
fm = final_sel["final_model"]
bench = final_sel["benchmarks"]

print("=" * 72)
print("ICE TRAIN DELAY PREDICTION — FINAL CLOSE-OUT")
print("=" * 72)
print(f"Project : {config['project']['title']}")
print(f"Authors : {', '.join(config['project']['authors'])}")
print()

checklist = []


def check(label: str, path: Path) -> bool:
    ok = path.exists()
    checklist.append({"label": label, "path": str(path), "exists": ok})
    print(f"  [{'OK' if ok else 'MISSING'}] {label}")
    return ok


print("DELIVERABLES CHECKLIST")
print("-" * 72)

all_ok = True
all_ok &= check("ice_train.parquet", PROCESSED_DIR / "ice_train.parquet")
all_ok &= check("ice_test.parquet", PROCESSED_DIR / "ice_test.parquet")
all_ok &= check("final_model.joblib", MODELS_DIR / "final_model.joblib")
all_ok &= check("final_model_selection.json", REFERENCE_DIR / "final_model_selection.json")
all_ok &= check("boosting_results.json", REFERENCE_DIR / "boosting_results.json")
all_ok &= check("full_project_pipeline_summary.json", PIPELINE_SUMMARY)
all_ok &= check("final_model_mae_comparison.png", FIGURES_DIR / "final_model_mae_comparison.png")

print()
print("FINAL RESULTS")
print("-" * 72)
print(f"  Final model      : {fm.get('export_name', fm['algorithm'])} ({fm['feature_set']})")
print(f"  Final test MAE   : {fm['test_mae']:.4f} min")
print(f"  Naive median MAE : {bench['naive_median_test_mae']:.4f} min")
print()
print("  Models compared:")
print("    Naive | Linear/Ridge/Lasso/ElasticNet")
print("    Random Forest (+ tuned) | GradientBoosting | XGBoost | LightGBM")
print()

if all_ok:
    print("ALL CORE DELIVERABLES: OK")
else:
    missing = [c["label"] for c in checklist if not c["exists"]]
    print(f"WARNING — missing: {missing}")

nb09_summary = {
    "metadata": {
        "notebook": "09_Final_Pipeline_Export",
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "PROJECT COMPLETE",
    },
    "checklist": checklist,
    "all_core_deliverables_ok": all_ok,
    "final_model_test_mae": fm["test_mae"],
    "naive_median_test_mae": bench["naive_median_test_mae"],
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(nb09_summary), f, indent=2, ensure_ascii=False)

project_complete = {
    "status": "COMPLETE",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_title": config["project"]["title"],
    "authors": config["project"]["authors"],
    "final_ml_model": f"{fm.get('export_name', fm['algorithm'])} ({fm['feature_set']})",
    "final_ml_test_mae_min": fm["test_mae"],
    "best_overall_test_mae_min": bench["naive_median_test_mae"],
    "one_line_conclusion": final_sel["honest_conclusion"],
}

with PROJECT_COMPLETE.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(project_complete), f, indent=2, ensure_ascii=False)

print()
print("=" * 72)
print("PROJECT COMPLETE")
print("=" * 72)
print(f"Saved: {SUMMARY_PATH}")
print(f"Saved: {PROJECT_COMPLETE}")
print("=" * 72)

ICE TRAIN DELAY PREDICTION — FINAL CLOSE-OUT
Project : ICE Train Delay Prediction Using Railway Operational Data and Historical Weather Data
Authors : Manikanta Engalligi, abhinay-sambherao

DELIVERABLES CHECKLIST
------------------------------------------------------------------------
  [OK] ice_train.parquet
  [OK] ice_test.parquet
  [OK] final_model.joblib
  [OK] final_model_selection.json
  [OK] boosting_results.json
  [OK] full_project_pipeline_summary.json
  [OK] final_model_mae_comparison.png

FINAL RESULTS
------------------------------------------------------------------------
  Final model      : XGBoost (full_with_weather)
  Final test MAE   : 10.3111 min
  Naive median MAE : 9.4813 min

  Models compared:
    Naive | Linear/Ridge/Lasso/ElasticNet
    Random Forest (+ tuned) | GradientBoosting | XGBoost | LightGBM

ALL CORE DELIVERABLES: OK

PROJECT COMPLETE
Saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\notebook_09_summary.json
Saved: c:\Users\Manikan